# Eksperimen 10: Pure LGBM Final Boss
**Strategi Kunci: Fokus Mutlak pada Algoritma Terkuat**

Berdasarkan rentetan eksperimen K-Fold Stacking sebelumnya, setiap model sekunder (XGBoost, CatBoost, ExtraTrees, HistBoost) yang digabungkan dengan LightGBM selalu menghasilkan bobot negatif atau mendekati nol pada Ridge Meta-Learner. Hal ini merupakan sinyal empiris yang sangat kuat bahwa LGBM adalah satu-satunya model yang benar-benar memahami pola data, sementara model lain sekadar menciptakan *noise* atau *bias*.

Eksperimen ini kembali ke himpunan 51 fitur terbaik (tanpa `steps_ahead` yang terbukti cacat secara desain untuk arsitektur eksogen), dan memusatkan 100% daya komputasi untuk mencari arsitektur LGBM yang sempurna.

In [1]:
import pandas as pd
import numpy as np
import warnings
import lightgbm as lgb
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error
from sklearn.cluster import KMeans
from sklearn.model_selection import TimeSeriesSplit
import optuna
import os

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 150)
optuna.logging.set_verbosity(optuna.logging.WARNING)

## 1. Pemuatan Data

In [2]:
train = pd.read_csv('../data/raw/train.csv')
test = pd.read_csv('../data/raw/test.csv')
env_data = pd.read_csv('../data/raw/data_pendukung/data_lingkungan.csv')
coords = pd.read_csv('../data/raw/data_pendukung/koordinat_pos.csv')

train['datetime'] = pd.to_datetime(train['datetime'])
test['datetime'] = pd.to_datetime(test['id'].str[:19])
test['nama_pos'] = test['id'].str[22:]
env_data['datetime'] = pd.to_datetime(env_data['datetime'])

## 2. EDA & CEDA

In [3]:
print("=== Dimensi Matriks ===")
print("Train:", train.shape)
print("Test:", test.shape)
print("Lingkungan:", env_data.shape)

print("\n=== Missing Values Data Lingkungan ===")
print(env_data.isnull().sum()[env_data.isnull().sum() > 0])

=== Dimensi Matriks ===
Train: (84396, 3)
Test: (21780, 3)
Lingkungan: (888480, 27)

=== Missing Values Data Lingkungan ===
soil_moisture_0_7cm          720
soil_moisture_7_28cm         720
soil_moisture_28_100cm       720
soil_moisture_100_255cm      720
surface_pressure_hpa         720
pressure_msl_hpa             720
rmm1                         720
rmm2                         720
mjo_phase                    720
mjo_amplitude                720
mjo_active                   720
nino_34                    12960
dtype: int64


### 2.1 Distribusi TMA per Pos & Last Known Condition

In [4]:
station_stats = train.groupby('nama_pos')['tma_mdpl'].agg(['mean', 'std', 'min', 'max'])
print("=== Distribusi TMA per Pos ===")
print(station_stats.sort_values('mean', ascending=False).round(2).to_string())

last_known = (
    train.sort_values('datetime')
    .groupby('nama_pos')
    .agg(tma_last_known=('tma_mdpl', 'last'))
    .reset_index()
)

=== Distribusi TMA per Pos ===
                             mean   std     min     max
nama_pos                                               
Ngadipiro                  143.56  0.33  143.18  146.31
Ngrembang                  140.05  0.29  139.85  143.79
Wonogiri Dam               132.34  3.30  125.54  137.36
Badegan                    122.40  0.31  121.90  123.96
Colo Weir                  107.79  1.12  102.19  109.70
Kali Pepe - Tugu Boto       94.82  0.45   94.35  100.42
Peren                       91.31  2.02   90.11  170.10
Jarum                       90.72  3.92   89.29  250.14
Sekayu                      87.27  0.66   86.61   92.32
Kali Anyar - Kreteg Abang   86.46  4.46   84.44  323.21
Serenan                     86.32  0.70   85.47   90.98
Kali Pepe - PTPN            82.55  2.00    0.00  138.07
Jurug                       78.82  1.94    0.00   86.46
Kedungupit                  64.87  3.08   62.13  207.74
Kajangan                    51.04  2.61   49.18  172.77
Ketonggo         

### 2.2 CEDA: Korelasi Variabel Lingkungan

In [5]:
train_temp = pd.merge(train, env_data, on=['datetime', 'nama_pos'], how='left')
num_cols = train_temp.select_dtypes(include=[np.number]).columns
corr = train_temp[num_cols].corr()['tma_mdpl'].sort_values(ascending=False)
print("=== Top Korelasi terhadap TMA ===")
print(corr.head(6).round(4))
print("...")
print(corr.tail(5).round(4))

=== Top Korelasi terhadap TMA ===
tma_mdpl                   1.0000
soil_moisture_100_255cm    0.1894
built_surface_m2           0.1793
soil_moisture_28_100cm     0.1270
soil_moisture_7_28cm       0.1238
soil_moisture_0_7cm        0.1058
Name: tma_mdpl, dtype: float64
...
rainfall_max_24h_mm    -0.0251
temperature_c          -0.0770
dew_point_c            -0.1217
landcover_class        -0.1573
surface_pressure_hpa   -0.9474
Name: tma_mdpl, dtype: float64


## 3. Station Profile & Last Known (Anti-Leakage)
Statistik per pos dihitung **murni dari train**. Last Known TMA digunakan sebagai sinyal awal kondisi sungai.

In [6]:
station_profile = train.groupby('nama_pos')['tma_mdpl'].agg(
    tma_mean='mean',
    tma_std='std',
    tma_p25=lambda x: x.quantile(0.25),
    tma_p75=lambda x: x.quantile(0.75)
).reset_index()
station_profile['tma_std'] = station_profile['tma_std'].fillna(1.0)

global_mean = train['tma_mdpl'].mean()
global_std = train['tma_mdpl'].std()

station_profile = pd.merge(station_profile, last_known, on='nama_pos', how='left')
station_profile['tma_last_known_norm'] = (
    (station_profile['tma_last_known'] - station_profile['tma_mean']) / station_profile['tma_std']
)

## 4. Preprocessing Adaptif

In [7]:
env_data = env_data.sort_values(['nama_pos', 'datetime'])

macro_cols = ['nino_34', 'mjo_phase', 'mjo_amplitude', 'mjo_active', 'rmm1', 'rmm2']
dynamic_cols = ['surface_pressure_hpa', 'pressure_msl_hpa', 'soil_moisture_0_7cm',
                'soil_moisture_7_28cm', 'soil_moisture_28_100cm', 'soil_moisture_100_255cm']

for c in macro_cols:
    env_data[c] = env_data.groupby('nama_pos')[c].ffill().bfill()

for c in dynamic_cols:
    env_data[c] = env_data.groupby('nama_pos')[c].apply(
        lambda x: x.interpolate(method='linear').bfill().ffill()
    ).reset_index(level=0, drop=True)

### 4.1 Agregasi & Penggabungan Spasial

In [8]:
kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
coords['spatial_cluster'] = kmeans.fit_predict(coords[['latitude', 'longitude']])

def aggregate_env_data(df):
    agg_funcs = {col: 'mean' for col in df.columns if col not in ['nama_pos', 'landcover_name', 'datetime']}
    agg_funcs['rainfall_mm'] = 'sum'
    agg_funcs['rainfall_openmeteo_mm'] = 'sum'
    agg_funcs['rainfall_max_24h_mm'] = 'max'
    df_indexed = df.set_index('datetime')
    return df_indexed.groupby(['nama_pos', pd.Grouper(freq='3h', label='right', closed='right')]).agg(agg_funcs).reset_index()

env_agg = aggregate_env_data(env_data)

test['tma_mdpl'] = np.nan
all_data = pd.concat([train, test], ignore_index=True)
all_data = all_data.sort_values(['nama_pos', 'datetime']).reset_index(drop=True)

all_data = pd.merge(all_data, env_agg, on=['datetime', 'nama_pos'], how='left')
all_data = pd.merge(all_data, coords, on='nama_pos', how='left')
all_data = pd.merge(all_data, station_profile, on='nama_pos', how='left')

all_data['tma_mean'] = all_data['tma_mean'].fillna(global_mean)
all_data['tma_std'] = all_data['tma_std'].fillna(global_std)
all_data['tma_last_known'] = all_data['tma_last_known'].fillna(global_mean)
all_data['tma_last_known_norm'] = all_data['tma_last_known_norm'].fillna(0)

## 5. Deep Feature Engineering
Kembali ke himpunan fitur terkuat kita. Fitur *rolling windows* menangkap rambatan fluida, sementara *last known* memberikan sinyal batas awal. Tidak ada *feature* kosmetik yang menyebabkan extrapolasi buta.

In [9]:
le = LabelEncoder()
all_data['nama_pos_encoded'] = le.fit_transform(all_data['nama_pos'])

all_data['month'] = all_data['datetime'].dt.month
all_data['hour'] = all_data['datetime'].dt.hour
all_data['day_of_year'] = all_data['datetime'].dt.dayofyear
all_data['sin_hour'] = np.sin(2 * np.pi * all_data['hour'] / 24)
all_data['cos_hour'] = np.cos(2 * np.pi * all_data['hour'] / 24)
all_data['sin_month'] = np.sin(2 * np.pi * all_data['month'] / 12)
all_data['cos_month'] = np.cos(2 * np.pi * all_data['month'] / 12)

all_data['runoff_factor'] = all_data['rainfall_mm'] * all_data['soil_moisture_0_7cm']
all_data['pressure_drop'] = all_data.groupby('nama_pos')['surface_pressure_hpa'].diff(1).fillna(0)
all_data['rainfall_intensity'] = all_data['rainfall_mm'] / (all_data['soil_moisture_0_7cm'] + 1e-6)
all_data['temp_humidity_index'] = all_data['temperature_c'] * all_data['humidity_pct'] / 100

windows = [4, 8, 24, 56]
for w in windows:
    all_data[f'rainfall_roll_{w}'] = all_data.groupby('nama_pos')['rainfall_mm'].transform(
        lambda x: x.rolling(window=w, min_periods=1).sum()
    )
    all_data[f'soil_roll_{w}'] = all_data.groupby('nama_pos')['soil_moisture_0_7cm'].transform(
        lambda x: x.rolling(window=w, min_periods=1).mean()
    )
    all_data[f'pressure_roll_{w}'] = all_data.groupby('nama_pos')['surface_pressure_hpa'].transform(
        lambda x: x.rolling(window=w, min_periods=1).mean()
    )
    all_data[f'temp_roll_{w}'] = all_data.groupby('nama_pos')['temperature_c'].transform(
        lambda x: x.rolling(window=w, min_periods=1).mean()
    )

## 6. Target Normalization per Pos

In [10]:
train_mask = all_data['tma_mdpl'].notnull()

all_data['tma_normalized'] = np.nan
all_data.loc[train_mask, 'tma_normalized'] = (
    (all_data.loc[train_mask, 'tma_mdpl'] - all_data.loc[train_mask, 'tma_mean']) /
    all_data.loc[train_mask, 'tma_std']
)

print("Distribusi target setelah normalisasi:")
print(all_data.loc[train_mask, 'tma_normalized'].describe().round(4))

Distribusi target setelah normalisasi:
count    84396.0000
mean         0.0000
std          0.9998
min        -41.2179
25%         -0.4486
50%         -0.1218
75%          0.2773
max         53.0309
Name: tma_normalized, dtype: float64


## 7. Training Setup
Membangun matriks pelatihan. Karena kita hanya melatih satu model, kita akan mengefisienkan seluruh resource Optuna di sini.

In [11]:
train_data = all_data[train_mask].sort_values('datetime').reset_index(drop=True)
test_data = all_data[~train_mask].sort_values('datetime').reset_index(drop=True)

drop_cols = ['datetime', 'nama_pos', 'tma_mdpl', 'tma_normalized', 'id', 'landcover_name']
features = [c for c in train_data.columns if c not in drop_cols]
target = 'tma_normalized'

X_full = train_data[features]
y_full = train_data[target]
X_test = test_data[features]

tscv = TimeSeriesSplit(n_splits=5)
print(f"Total fitur yang digunakan: {len(features)}")
print(features)

Total fitur yang digunakan: 61
['rainfall_mm', 'humidity_pct', 'wind_direction_deg', 'dew_point_c', 'cloud_cover_pct', 'temperature_c', 'wind_speed_kmh', 'rainfall_openmeteo_mm', 'rainfall_max_24h_mm', 'solar_radiation_mj_m2', 'soil_moisture_0_7cm', 'soil_moisture_7_28cm', 'soil_moisture_28_100cm', 'soil_moisture_100_255cm', 'surface_pressure_hpa', 'pressure_msl_hpa', 'built_surface_m2', 'landcover_class', 'rmm1', 'rmm2', 'mjo_phase', 'mjo_amplitude', 'mjo_active', 'nino_34', 'latitude', 'longitude', 'spatial_cluster', 'tma_mean', 'tma_std', 'tma_p25', 'tma_p75', 'tma_last_known', 'tma_last_known_norm', 'nama_pos_encoded', 'month', 'hour', 'day_of_year', 'sin_hour', 'cos_hour', 'sin_month', 'cos_month', 'runoff_factor', 'pressure_drop', 'rainfall_intensity', 'temp_humidity_index', 'rainfall_roll_4', 'soil_roll_4', 'pressure_roll_4', 'temp_roll_4', 'rainfall_roll_8', 'soil_roll_8', 'pressure_roll_8', 'temp_roll_8', 'rainfall_roll_24', 'soil_roll_24', 'pressure_roll_24', 'temp_roll_24', 

### 7.1 Max Power Optuna Tuning: 75 Trials
Kita mengizinkan `n_estimators` hingga 5000, tetapi menggunakan `early_stopping_rounds=200` pada setiap fold. Ruang eksplorasi hiperparameter sangat agresif untuk mencegah overfitting sekaligus memaksimalkan kedalaman pola.

In [12]:
def objective_lgb(trial):
    params = {
        'n_estimators': 5000,
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.05, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 31, 255),
        'max_depth': trial.suggest_int('max_depth', 5, 12),
        'subsample': trial.suggest_float('subsample', 0.5, 0.9),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 0.9),
        'min_child_samples': trial.suggest_int('min_child_samples', 30, 300),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.1, 20.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.1, 20.0, log=True),
        'subsample_freq': 1,
        'random_state': 42,
        'verbose': -1
    }
    scores = []
    for train_idx, val_idx in tscv.split(X_full):
        X_tr, X_va = X_full.iloc[train_idx], X_full.iloc[val_idx]
        y_tr, y_va = y_full.iloc[train_idx], y_full.iloc[val_idx]
        m = lgb.LGBMRegressor(**params)
        m.fit(
            X_tr, y_tr,
            eval_set=[(X_va, y_va)],
            callbacks=[lgb.early_stopping(stopping_rounds=200, verbose=False)]
        )
        scores.append(mean_squared_error(y_va, m.predict(X_va)))
    return np.mean(scores)

print("Memulai Max Power Optuna Tuning (75 trials)...")
study_lgb = optuna.create_study(direction='minimize')
study_lgb.optimize(objective_lgb, n_trials=75)
best_lgb = study_lgb.best_params
best_lgb.update({'n_estimators': 5000, 'random_state': 42, 'verbose': -1, 'subsample_freq': 1})
print(f"\nBest LightGBM Params: {best_lgb}")

Memulai Max Power Optuna Tuning (75 trials)...

Best LightGBM Params: {'learning_rate': 0.04985789500024071, 'num_leaves': 60, 'max_depth': 6, 'subsample': 0.8870792997800095, 'colsample_bytree': 0.530214012394518, 'min_child_samples': 112, 'reg_alpha': 0.11704799750380937, 'reg_lambda': 2.4493601434231995, 'n_estimators': 5000, 'random_state': 42, 'verbose': -1, 'subsample_freq': 1}


## 8. Final K-Fold Training & Prediction
Melatih LGBM terbaik melintasi 5 lipatan, murni melaporkan rata-rata K-Fold. Prediksi *test* diambil rata-rata dari ke-5 fold.

In [13]:
test_preds_lgb = np.zeros(len(X_test))
cv_rmse_scores = []

print("Memulai K-Fold Final Training...")
for fold, (train_idx, val_idx) in enumerate(tscv.split(X_full)):
    X_tr, X_va = X_full.iloc[train_idx], X_full.iloc[val_idx]
    y_tr, y_va = y_full.iloc[train_idx], y_full.iloc[val_idx]

    val_mean = train_data.iloc[val_idx]['tma_mean'].values
    val_std = train_data.iloc[val_idx]['tma_std'].values

    m_lgb = lgb.LGBMRegressor(**best_lgb)
    m_lgb.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
        callbacks=[lgb.early_stopping(stopping_rounds=200, verbose=False)]
    )

    p_norm = m_lgb.predict(X_va)
    p_abs = (p_norm * val_std) + val_mean
    y_abs = (y_va.values * val_std) + val_mean

    fold_rmse = np.sqrt(mean_squared_error(y_abs, p_abs))
    cv_rmse_scores.append(fold_rmse)
    print(f"Fold {fold+1} RMSE (abs): {fold_rmse:.4f} | Trees Matang: {m_lgb.best_iteration_}")

    test_preds_lgb += m_lgb.predict(X_test) / tscv.n_splits

print(f"\nRata-rata K-Fold RMSE (Pure LGBM): {np.mean(cv_rmse_scores):.4f}")

Memulai K-Fold Final Training...
Fold 1 RMSE (abs): 3.7598 | Trees Matang: 211
Fold 2 RMSE (abs): 1.8125 | Trees Matang: 18
Fold 3 RMSE (abs): 0.6801 | Trees Matang: 57
Fold 4 RMSE (abs): 1.3009 | Trees Matang: 274
Fold 5 RMSE (abs): 1.0120 | Trees Matang: 133

Rata-rata K-Fold RMSE (Pure LGBM): 1.7130


## 9. Denormalisasi & Output Submisi

In [14]:
test_mean = test_data['tma_mean'].values
test_std = test_data['tma_std'].values
final_tma = (test_preds_lgb * test_std) + test_mean

test_data['tma_mdpl'] = final_tma
submission = test_data[['id', 'tma_mdpl']]

if not os.path.exists('../submissions'):
    os.makedirs('../submissions')

submission.to_csv('../submissions/submission.csv', index=False)
print("Eksperimen 10 selesai. File submission.csv tersimpan.")
print(f"Rentang prediksi TMA: {final_tma.min():.2f} - {final_tma.max():.2f}")

Eksperimen 10 selesai. File submission.csv tersimpan.
Rentang prediksi TMA: 0.77 - 144.43
